In [23]:
# import time
# time.sleep(2100)




---

### **4. Manejo de Datos Geoespaciales.ipynb**
  - **4.1. Introducción a niveles geográficos:** Descripción de los niveles geográficos disponibles.
  - **4.2. Carga y procesamiento de geometrías:** 
    - **4.2.1.** Importación de datos de polígonos.
    - **4.2.2.** Ajuste del Sistema de Referencia de Coordenadas (CRS).
    - **4.2.3.** Cálculo del área en km^2.
  - **4.3. Guardado de datos geoespaciales:** 
    - **4.3.1.** Funciones para guardar datos en formato GeoJSON.
    - **4.3.2.** Procesamiento y guardado de datos en diferentes niveles geográficos.

---

**CONTENIDO**

- Cargar data CS
- Resultados Complejos observables en CS
- Procesamiento para datasets geograficos
    - Mapas Ejemplo (Base Personas)
    - Comandos Mapbox CLI

In [32]:
import geopandas as gpd
import numpy as np
from funciones import generate_Qs_from_year
from datetime import datetime
import pandas as pd

# -------------------
# Parameters and Configuration
# -------------------

FRAC = 0.05
# START_YEAR = 2023
# END_YEAR = 2025
EXPERIMENT_TAG = 'ARG'

PATH_POBREZA = './../data/Pobreza/'


In [25]:


# Cargar el índice de precios al consumidor (CPI) desde la fuente
cpi = pd.read_csv('https://raw.githubusercontent.com/matuteiglesias/IPC-Argentina/main/data/info/indice_precios_M.csv', index_col=0)
cpi.index = pd.to_datetime(cpi.index)

# Obtener la fecha de hoy en formato año-mes
hoy = datetime.today().strftime('%Y-%m')

# Calcular el ratio de precios de hoy con respecto a los precios con índice en base al modelo
ix = cpi.loc[hoy, 'index'].values[0] / cpi.loc['2016-01', 'index'].values[0]

# Lista de columnas relacionadas con montos en pesos
columnas_pesos = ['P47T_persona', 'P47T_hogar', 'CBA', 'gap_indigencia', 'CBT', 'gap_pobreza']


In [26]:

# data = pd.read_csv('/media/matias/Elements/suite/indice-pobreza-ExactasUBA/data/Pobreza/pobreza_0.02_ARGCSactual.csv', encoding_errors='ignore')
#                     # /media/matias/Elements/suite/indice-pobreza-ExactasUBA/data/Pobreza/pobreza_0.02_ARGCSactual.csv
# ## Deflacta a precios actuales

# # Lista de fechas Q que se van a procesar
# Qs = ['2022-05-15', '2022-08-15', '2022-11-15', '2023-02-15']

# Listas para almacenar los datos consolidados de cada Q
all_info_personas = []
all_info_hogares = []

# Loop through the years
for yr in range(2023, 2025):
    yr_str = str(yr)  # Convert year to string for consistency with quarters
    print(f"Processing data for year {yr_str}...")



    # Load geographic data once for the year
    hogares_geo_file = f'{PATH_POBREZA}geo_households_sample{FRAC}_{yr_str}_{EXPERIMENT_TAG}.csv'
    try:
        hogares_geo = pd.read_csv(hogares_geo_file)
        print(f"Geographic data loaded for year {yr_str}.")
    except FileNotFoundError:
        print(f"Warning: Geographic data file {hogares_geo_file} not found. Skipping year {yr_str}.")
        continue


# /home/matias/repos/indice-pobreza-UBA/data/Pobreza/geo_households_sample0.02_2005_ARG.csv

    # Processing data for each quarter of the specified year
    relevant_quarters = generate_Qs_from_year(yr_str)
    for Q in relevant_quarters:
        if Q in ['2023-05-15', '2023-08-15', '2023-11-15', '2024-02-15']:
            print(f"Procesando fecha Q: {Q}")
            # /home/matias/repos/indice-pobreza-UBA/data/Pobreza/individual_income_sample0.02_2014-05-15_ARG.csv
            # /home/matias/repos/indice-pobreza-UBA/data/Pobreza/household_poverty_sample0.02_q2017-02-15.csv

            # Define the filenames for the quarter
            personas_ingresos_Q_file = f'{PATH_POBREZA}individual_income_sample{FRAC}_q{Q}_{EXPERIMENT_TAG}.csv'
            pobreza_hogares_Q_file = f'{PATH_POBREZA}household_poverty_sample{FRAC}_q{Q}_{EXPERIMENT_TAG}.csv'

            try:
                # Load the files for the current quarter
                personas_ingresos_Q = pd.read_csv(personas_ingresos_Q_file)
                pobreza_hogares = pd.read_csv(pobreza_hogares_Q_file)
            except FileNotFoundError as e:
                print(f"Warning: {e.filename} not found. Skipping quarter {Q}.")
                continue

            # Merge datasets to get the consolidated data
            info_personas = personas_ingresos_Q.merge(pobreza_hogares, on=['HOGAR_REF_ID', 'Q'], how='left').merge(hogares_geo, on='HOGAR_REF_ID', how='left')
            info_hogares = pobreza_hogares.merge(hogares_geo, on='HOGAR_REF_ID', how='left')

            # Append the consolidated datasets to the lists
            all_info_personas.append(info_personas)
            all_info_hogares.append(info_hogares)


Processing data for year 2023...


/tmp/ipykernel_57395/3720726655.py:22: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  hogares_geo = pd.read_csv(hogares_geo_file)


Geographic data loaded for year 2023.
Procesando fecha Q: 2023-05-15
Procesando fecha Q: 2023-08-15
Procesando fecha Q: 2023-11-15
Processing data for year 2024...


/tmp/ipykernel_57395/3720726655.py:22: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  hogares_geo = pd.read_csv(hogares_geo_file)


Geographic data loaded for year 2024.
Procesando fecha Q: 2024-02-15


In [27]:
 
# Concatenar todos los conjuntos de datos de las diferentes fechas
cross_section_personas = pd.concat(all_info_personas, ignore_index=True)
cross_section_hogares = pd.concat(all_info_hogares, ignore_index=True)


## Adaptar datos (Pesos actuales, AGLOS si, IDFRAC)
for df in [cross_section_personas, cross_section_hogares]:
    for col in columnas_pesos:
        if col in df.columns: 
            if col == 'P47T_persona': df[col] = np.power(10, df[col]) - 1
            df[col] = (ix*df[col]).round(-1).astype(int)
            
    # df = df.merge(radio_ref, axis = 1), on = ['RADIO_REF_ID', 'AGLOMERADO'], how = 'left')
    df['AGLOSI'] = df.AGLOMERADO != 0

    df['IDFRAC'] = df['COD_2010'].astype(str).str.zfill(9).str[:-2] + '00'


# -------------------


In [28]:
import geopandas as gpd

def compute_area_km2(geo_df):
    """Calcula el área en km^2 de un GeoDataFrame y lo añade como una nueva columna."""
    geo_df['area_km2'] = geo_df['geometry'].to_crs('epsg:3395').map(lambda p: p.area / 10**6)
    return geo_df

# Rutas de los archivos de polígonos
admin310_f = './../../geoespacial-censo-IGN/censos_shp_CONICET_dissolved/fracs_2010.shp'
admin210_f = './../../geoespacial-censo-IGN/censos_shp_CONICET_dissolved/dptos_2010.shp'
admin1_f = './../../geoespacial-censo-IGN/IGN_shp/ign_provincia'

# Cargar polígonos de provincias del IGN
admin1 = gpd.read_file(admin1_f)
admin1['PROV'] = admin1.IN1.astype(int)
admin1 = admin1[['PROV', 'geometry']]

# Cargar y procesar polígonos de fracciones de CONICET
admin310 = gpd.read_file(admin310_f)
admin310['IDFRAC'] = admin310.PROV_ + admin310.DEPTO_ + admin310.FRACC_ + '00'
admin310 = admin310[['IDFRAC', 'geometry']]

# Cargar y procesar polígonos de departamentos de CONICET
admin210 = gpd.read_file(admin210_f)
admin210['DPTO'] = (admin210['PROV_'] + admin210['DEPTO_']).astype(int)
admin210 = admin210[['DPTO', 'geometry']]

# Ajustar CRS para que todos los GeoDataFrames tengan el mismo CRS que admin1
admin210 = admin210.to_crs(admin1.crs)
admin310 = admin310.to_crs(admin1.crs)

# Calcular el área en km^2 para cada GeoDataFrame
admin1 = compute_area_km2(admin1)
admin210 = compute_area_km2(admin210)
admin310 = compute_area_km2(admin310)


/home/matias/anaconda3/envs/base2/lib/python3.11/site-packages/shapely/measurement.py:44: RuntimeWarning: invalid value encountered in area
  return lib.area(geometry, **kwargs)


In [29]:
from funciones import process_and_save
import os

In [30]:
# from funciones import * 


In [31]:
# Define los GeoDataFrames para cada nivel geográfico
geo_dfs = {
    'PROV': admin1,
    'DPTO': admin210,
    'IDFRAC': admin310
}

# Define los subconjuntos de datos y sus prefijos correspondientes para los nombres de los archivos
data_subsets = {
    'P': cross_section_personas,
    'M24': cross_section_personas[cross_section_personas.P03 >= 24],
    'M14': cross_section_personas[cross_section_personas.P03 <= 14],
    'M6': cross_section_personas[cross_section_personas.P03 <= 6],
    'H': cross_section_hogares  # Asumiendo que data también incluye información de hogares
}

ow = False  # Sobrescribir archivos existentes
# Procesar y guardar los datos para cada subconjunto y nivel geográfico
for filename_prefix, subset in data_subsets.items():
    for geo_level, geo_df in geo_dfs.items():
        print(f"Procesando subset '{filename_prefix}' a nivel geografico '{geo_level}'...")
        
        if not ow or os.path.exists(f'./../data/geojson/poverty_{filename_prefix}_{geo_level}_sample{FRAC}.geojson'): continue
        
        process_and_save(subset, geo_level, geo_df, filename_prefix)
        print(f"Procesamiento de subset '{filename_prefix}' a nivel geografico '{geo_level}' completado.")
    print(f"Procesamiento completo para subset '{filename_prefix}'.")
print("Todos los procesamientos han sido completados.")



Procesando subset 'P' a nivel geografico 'PROV'...
Procesando subset 'P' a nivel geografico 'DPTO'...
Procesando subset 'P' a nivel geografico 'IDFRAC'...
Procesamiento completo para subset 'P'.
Procesando subset 'M24' a nivel geografico 'PROV'...
Procesando subset 'M24' a nivel geografico 'DPTO'...
Procesando subset 'M24' a nivel geografico 'IDFRAC'...
Procesamiento completo para subset 'M24'.
Procesando subset 'M14' a nivel geografico 'PROV'...
Procesando subset 'M14' a nivel geografico 'DPTO'...
Procesando subset 'M14' a nivel geografico 'IDFRAC'...
Procesamiento completo para subset 'M14'.
Procesando subset 'M6' a nivel geografico 'PROV'...
Procesando subset 'M6' a nivel geografico 'DPTO'...
Procesando subset 'M6' a nivel geografico 'IDFRAC'...
Procesamiento completo para subset 'M6'.
Procesando subset 'H' a nivel geografico 'PROV'...
Procesando subset 'H' a nivel geografico 'DPTO'...
Procesando subset 'H' a nivel geografico 'IDFRAC'...
Procesamiento completo para subset 'H'.
Todos